## Trạng thái bản lưu trên GitHub

Baseline lịch sử; không phải model web đang triển khai.

Output cell đã được bỏ để tránh file tạm, dữ liệu nhạy cảm và bảng lỗi cũ. Mã cell được giữ; kết quả kiểm chứng nằm trong thư mục `results/`. Xem `docs/notebook-guide.md` trước khi chạy. Không cần chạy tuần tự cả tám notebook.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import copy
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    DataLoader,
    TensorDataset,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)


print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

In [ ]:
def find_kaggle_file(filename):
    """Tìm file trong working hoặc notebook input."""
    working_path = (
        Path("/kaggle/working") / filename
    )

    if working_path.exists():
        return working_path

    candidates = sorted(
        Path("/kaggle/input").rglob(filename)
    )

    if not candidates:
        raise FileNotFoundError(
            f"Không tìm thấy {filename}"
        )

    if len(candidates) > 1:
        print(
            "Tìm thấy nhiều file, đang dùng:",
            candidates[0],
        )

    return candidates[0]


FEATURE_PATH = find_kaggle_file(
    "gmdcsa24_features_32.npz"
)


with np.load(FEATURE_PATH) as data:
    skeleton_features = data[
        "skeleton_features"
    ].astype(np.float32)

    labels = data[
        "labels"
    ].astype(np.int64)

    subjects = data[
        "subjects"
    ].astype(np.int64)

    video_ids = data[
        "video_ids"
    ].astype("U")


# skeleton_features: x, y, z, visibility
# Vanilla ST-GCN chỉ lấy x, y, visibility
stgcn_input = skeleton_features[
    ...,
    [0, 1, 3],
]


# Từ N,T,V,C sang N,C,T,V
stgcn_input = np.transpose(
    stgcn_input,
    (0, 3, 1, 2),
).astype(np.float32)


assert stgcn_input.shape == (
    160,
    3,
    32,
    33,
)

assert np.isfinite(stgcn_input).all()


all_subjects = sorted(
    np.unique(subjects).tolist()
)


print("Feature path:", FEATURE_PATH)
print("ST-GCN input:", stgcn_input.shape)
print("Labels:", labels.shape)
print("Subjects:", all_subjects)
print("Finite:", np.isfinite(stgcn_input).all())

In [ ]:
NUM_JOINTS = 33


# Các cạnh của MediaPipe Pose
POSE_EDGES = [
    # Khuôn mặt
    (0, 1),
    (1, 2),
    (2, 3),
    (3, 7),

    (0, 4),
    (4, 5),
    (5, 6),
    (6, 8),

    (0, 9),
    (0, 10),
    (9, 10),

    # Nối đầu với hai vai
    (0, 11),
    (0, 12),

    # Vai và tay trái
    (11, 12),
    (11, 13),
    (13, 15),
    (15, 17),
    (15, 19),
    (15, 21),
    (17, 19),

    # Tay phải
    (12, 14),
    (14, 16),
    (16, 18),
    (16, 20),
    (16, 22),
    (18, 20),

    # Thân người
    (11, 23),
    (12, 24),
    (23, 24),

    # Chân trái
    (23, 25),
    (25, 27),
    (27, 29),
    (29, 31),
    (27, 31),

    # Chân phải
    (24, 26),
    (26, 28),
    (28, 30),
    (30, 32),
    (28, 32),
]


# Thêm self-loop
adjacency = np.eye(
    NUM_JOINTS,
    dtype=np.float32,
)


# Đồ thị vô hướng
for start_joint, end_joint in POSE_EDGES:
    adjacency[
        start_joint,
        end_joint,
    ] = 1.0

    adjacency[
        end_joint,
        start_joint,
    ] = 1.0


# Chuẩn hóa đối xứng: D^(-1/2) A D^(-1/2)
degree = adjacency.sum(axis=1)

inverse_sqrt_degree = np.power(
    np.maximum(degree, 1e-6),
    -0.5,
)

degree_matrix = np.diag(
    inverse_sqrt_degree
)

normalized_adjacency = (
    degree_matrix
    @ adjacency
    @ degree_matrix
).astype(np.float32)


ADJACENCY = torch.tensor(
    normalized_adjacency,
    dtype=torch.float32,
)


assert ADJACENCY.shape == (33, 33)
assert torch.isfinite(ADJACENCY).all()


print("Adjacency:", ADJACENCY.shape)
print("Số cạnh cơ thể:", len(POSE_EDGES))
print(
    "Đối xứng:",
    torch.allclose(
        ADJACENCY,
        ADJACENCY.T,
    ),
)

In [ ]:
BASE_SEED = 42


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def create_loader(
    indices,
    batch_size,
    shuffle,
    seed,
):
    input_tensor = torch.from_numpy(
        stgcn_input[indices]
    )

    label_tensor = torch.from_numpy(
        labels[indices]
    )

    dataset = TensorDataset(
        input_tensor,
        label_tensor,
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=generator,
    )


def calculate_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    false_alarm_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0.0
    )

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "sensitivity": recall_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "specificity": specificity,
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "false_alarm_rate": false_alarm_rate,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


set_seed(BASE_SEED)

print("Seed:", BASE_SEED)

In [ ]:
class SpatialGraphConv(nn.Module):
    def __init__(
        self,
        input_channels,
        output_channels,
    ):
        super().__init__()

        self.projection = nn.Conv2d(
            input_channels,
            output_channels,
            kernel_size=1,
        )

    def forward(
        self,
        features,
        adjacency_matrix,
    ):
        # Truyền thông tin giữa các khớp nối nhau
        features = torch.einsum(
            "nctv,vw->nctw",
            features,
            adjacency_matrix,
        )

        return self.projection(features)


class STGCNBlock(nn.Module):
    def __init__(
        self,
        input_channels,
        output_channels,
        temporal_stride=1,
        dropout=0.3,
        residual=True,
    ):
        super().__init__()

        self.spatial_graph_conv = (
            SpatialGraphConv(
                input_channels,
                output_channels,
            )
        )

        self.temporal_conv = nn.Sequential(
            nn.BatchNorm2d(
                output_channels
            ),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                output_channels,
                output_channels,
                kernel_size=(9, 1),
                stride=(temporal_stride, 1),
                padding=(4, 0),
            ),

            nn.BatchNorm2d(
                output_channels
            ),

            nn.Dropout(dropout),
        )

        if not residual:
            self.residual = None

        elif (
            input_channels == output_channels
            and temporal_stride == 1
        ):
            self.residual = nn.Identity()

        else:
            self.residual = nn.Sequential(
                nn.Conv2d(
                    input_channels,
                    output_channels,
                    kernel_size=1,
                    stride=(
                        temporal_stride,
                        1,
                    ),
                ),
                nn.BatchNorm2d(
                    output_channels
                ),
            )

        self.activation = nn.ReLU(
            inplace=True
        )

    def forward(
        self,
        features,
        adjacency_matrix,
    ):
        residual = (
            0
            if self.residual is None
            else self.residual(features)
        )

        features = self.spatial_graph_conv(
            features,
            adjacency_matrix,
        )

        features = self.temporal_conv(
            features
        )

        return self.activation(
            features + residual
        )


class VanillaSTGCN(nn.Module):
    def __init__(
        self,
        adjacency_matrix,
        input_channels=3,
        num_classes=2,
        dropout=0.3,
    ):
        super().__init__()

        self.register_buffer(
            "adjacency",
            adjacency_matrix.clone(),
        )

        # Chuẩn hóa đầu vào C × V
        self.input_batch_norm = nn.BatchNorm1d(
            input_channels * NUM_JOINTS
        )

        self.blocks = nn.ModuleList([
            STGCNBlock(
                input_channels,
                64,
                dropout=dropout,
                residual=False,
            ),
            STGCNBlock(
                64,
                64,
                dropout=dropout,
            ),
            STGCNBlock(
                64,
                128,
                temporal_stride=2,
                dropout=dropout,
            ),
            STGCNBlock(
                128,
                128,
                dropout=dropout,
            ),
        ])

        # Cho phép học lại trọng số các cạnh
        self.edge_importance = (
            nn.ParameterList([
                nn.Parameter(
                    torch.ones_like(
                        adjacency_matrix
                    )
                )
                for _ in self.blocks
            ])
        )

        self.classifier = nn.Linear(
            128,
            num_classes,
        )

    def forward(self, features):
        # Input: N, C, T, V
        batch_size, channels, frames, joints = (
            features.shape
        )

        # N,C,T,V -> N,V*C,T
        features = (
            features
            .permute(0, 3, 1, 2)
            .contiguous()
            .view(
                batch_size,
                joints * channels,
                frames,
            )
        )

        features = self.input_batch_norm(
            features
        )

        # Quay về N,C,T,V
        features = (
            features
            .view(
                batch_size,
                joints,
                channels,
                frames,
            )
            .permute(0, 2, 3, 1)
            .contiguous()
        )

        for block, importance in zip(
            self.blocks,
            self.edge_importance,
        ):
            features = block(
                features,
                self.adjacency * importance,
            )

        # Global average pooling theo T và V
        features = features.mean(
            dim=(2, 3)
        )

        return self.classifier(features)


set_seed(BASE_SEED)

test_model = VanillaSTGCN(
    ADJACENCY
).to(DEVICE)


sample_tensor = torch.from_numpy(
    stgcn_input[:2]
).to(DEVICE)


with torch.no_grad():
    sample_output = test_model(
        sample_tensor
    )


parameter_count = sum(
    parameter.numel()
    for parameter in test_model.parameters()
)


print("Input:", sample_tensor.shape)
print("Output:", sample_output.shape)
print(
    "Số tham số:",
    f"{parameter_count:,}",
)


del test_model
del sample_tensor

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 120
MIN_EPOCHS = 20
PATIENCE = 20

DROPOUT = 0.3


def train_one_epoch(
    model,
    data_loader,
    optimizer,
    criterion,
):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for inputs, targets in data_loader:
        inputs = inputs.to(
            DEVICE,
            non_blocking=True,
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(inputs)

        loss = criterion(
            logits,
            targets,
        )

        loss.backward()
        optimizer.step()

        batch_size = targets.size(0)

        total_loss += (
            loss.item() * batch_size
        )

        total_samples += batch_size

    return total_loss / total_samples


@torch.no_grad()
def evaluate_model(
    model,
    data_loader,
    criterion,
):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    true_labels = []
    predicted_labels = []
    fall_probabilities = []

    for inputs, targets in data_loader:
        inputs = inputs.to(
            DEVICE,
            non_blocking=True,
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True,
        )

        logits = model(inputs)

        loss = criterion(
            logits,
            targets,
        )

        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        predictions = logits.argmax(
            dim=1
        )

        batch_size = targets.size(0)

        total_loss += (
            loss.item() * batch_size
        )

        total_samples += batch_size

        true_labels.extend(
            targets.cpu().numpy()
        )

        predicted_labels.extend(
            predictions.cpu().numpy()
        )

        fall_probabilities.extend(
            probabilities[:, 1]
            .cpu()
            .numpy()
        )

    true_labels = np.asarray(
        true_labels,
        dtype=np.int64,
    )

    predicted_labels = np.asarray(
        predicted_labels,
        dtype=np.int64,
    )

    fall_probabilities = np.asarray(
        fall_probabilities,
        dtype=np.float32,
    )

    metrics = calculate_metrics(
        true_labels,
        predicted_labels,
    )

    return (
        total_loss / total_samples,
        metrics,
        true_labels,
        predicted_labels,
        fall_probabilities,
    )


def train_with_early_stopping(
    train_indices,
    val_indices,
    seed,
):
    set_seed(seed)

    model = VanillaSTGCN(
        ADJACENCY,
        dropout=DROPOUT,
    ).to(DEVICE)

    train_loader = create_loader(
        train_indices,
        BATCH_SIZE,
        shuffle=True,
        seed=seed,
    )

    val_loader = create_loader(
        val_indices,
        BATCH_SIZE,
        shuffle=False,
        seed=seed,
    )

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_state = None
    best_epoch = 0
    best_metrics = None

    # Balanced accuracy, F1, sau đó loss
    best_score = (
        -np.inf,
        -np.inf,
        -np.inf,
    )

    waiting_epochs = 0
    history = []

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
        )

        (
            val_loss,
            val_metrics,
            _,
            _,
            _,
        ) = evaluate_model(
            model,
            val_loader,
            criterion,
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            **val_metrics,
        })

        current_score = (
            val_metrics[
                "balanced_accuracy"
            ],
            val_metrics["f1"],
            -val_loss,
        )

        if current_score > best_score:
            best_score = current_score
            best_epoch = epoch
            best_metrics = (
                val_metrics.copy()
            )

            best_state = copy.deepcopy(
                model.state_dict()
            )

            waiting_epochs = 0

        else:
            waiting_epochs += 1

        if (
            epoch >= MIN_EPOCHS
            and waiting_epochs >= PATIENCE
        ):
            break

    model.load_state_dict(
        best_state
    )

    return (
        model,
        best_epoch,
        best_metrics,
        pd.DataFrame(history),
    )


def train_fixed_epochs(
    train_indices,
    number_of_epochs,
    seed,
):
    set_seed(seed)

    model = VanillaSTGCN(
        ADJACENCY,
        dropout=DROPOUT,
    ).to(DEVICE)

    train_loader = create_loader(
        train_indices,
        BATCH_SIZE,
        shuffle=True,
        seed=seed,
    )

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    training_losses = []

    for epoch in range(
        1,
        number_of_epochs + 1,
    ):
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
        )

        training_losses.append({
            "epoch": epoch,
            "train_loss": train_loss,
        })

    return (
        model,
        pd.DataFrame(training_losses),
    )


print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Max epochs:", MAX_EPOCHS)
print("Patience:", PATIENCE)

In [ ]:
OUTPUT_DIR = Path(
    "/kaggle/working/"
    "vanilla_stgcn_results"
)

MODEL_DIR = Path(
    "/kaggle/working/"
    "vanilla_stgcn_models"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


inner_result_records = []
outer_result_records = []
prediction_records = []


for test_subject in all_subjects:
    print("\n" + "=" * 60)
    print(
        f"OUTER TEST SUBJECT: S{test_subject}"
    )
    print("=" * 60)

    development_subjects = [
        subject
        for subject in all_subjects
        if subject != test_subject
    ]

    inner_best_epochs = []

    # Ba inner folds
    for val_subject in development_subjects:
        train_subjects = [
            subject
            for subject in development_subjects
            if subject != val_subject
        ]

        train_indices = np.where(
            np.isin(
                subjects,
                train_subjects,
            )
        )[0]

        val_indices = np.where(
            subjects == val_subject
        )[0]

        current_seed = (
            BASE_SEED
            + test_subject * 100
            + val_subject
        )

        (
            inner_model,
            best_epoch,
            best_metrics,
            inner_history,
        ) = train_with_early_stopping(
            train_indices,
            val_indices,
            seed=current_seed,
        )

        inner_best_epochs.append(
            best_epoch
        )

        inner_result_records.append({
            "test_subject": test_subject,
            "val_subject": val_subject,
            "train_subjects": ",".join(
                map(str, train_subjects)
            ),
            "train_samples": len(
                train_indices
            ),
            "val_samples": len(
                val_indices
            ),
            "best_epoch": best_epoch,
            **best_metrics,
        })

        history_path = (
            OUTPUT_DIR
            / (
                f"history_outer_{test_subject}"
                f"_val_{val_subject}.csv"
            )
        )

        inner_history.to_csv(
            history_path,
            index=False,
        )

        print(
            f"Inner val S{val_subject}"
            f" | train {train_subjects}"
            f" | best epoch {best_epoch}"
            f" | balanced accuracy "
            f"{best_metrics['balanced_accuracy']:.4f}"
        )

        del inner_model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Không dùng test để chọn epoch
    final_epochs = int(
        round(
            np.median(
                inner_best_epochs
            )
        )
    )

    final_epochs = max(
        final_epochs,
        1,
    )

    print(
        "Inner best epochs:",
        inner_best_epochs,
    )

    print(
        "Final training epochs:",
        final_epochs,
    )

    # Train lại bằng toàn bộ ba subject
    final_train_indices = np.where(
        subjects != test_subject
    )[0]

    test_indices = np.where(
        subjects == test_subject
    )[0]

    final_seed = (
        BASE_SEED + test_subject
    )

    (
        final_model,
        final_training_history,
    ) = train_fixed_epochs(
        final_train_indices,
        final_epochs,
        seed=final_seed,
    )

    test_loader = create_loader(
        test_indices,
        BATCH_SIZE,
        shuffle=False,
        seed=final_seed,
    )

    criterion = nn.CrossEntropyLoss()

    (
        test_loss,
        test_metrics,
        test_true,
        test_prediction,
        fall_probabilities,
    ) = evaluate_model(
        final_model,
        test_loader,
        criterion,
    )

    outer_result_records.append({
        "test_subject": test_subject,
        "train_subjects": ",".join(
            map(
                str,
                development_subjects,
            )
        ),
        "inner_best_epochs": ",".join(
            map(
                str,
                inner_best_epochs,
            )
        ),
        "final_epochs": final_epochs,
        "train_samples": len(
            final_train_indices
        ),
        "test_samples": len(
            test_indices
        ),
        "test_loss": test_loss,
        **test_metrics,
    })

    for local_index, sample_index in enumerate(
        test_indices
    ):
        prediction_records.append({
            "sample_index": int(
                sample_index
            ),
            "video_id": video_ids[
                sample_index
            ],
            "subject": int(
                subjects[sample_index]
            ),
            "true_label": int(
                test_true[local_index]
            ),
            "predicted_label": int(
                test_prediction[local_index]
            ),
            "fall_probability": float(
                fall_probabilities[
                    local_index
                ]
            ),
            "correct": bool(
                test_true[local_index]
                == test_prediction[
                    local_index
                ]
            ),
        })

    final_training_history.to_csv(
        OUTPUT_DIR
        / (
            f"final_history_test_"
            f"subject_{test_subject}.csv"
        ),
        index=False,
    )

    model_path = (
        MODEL_DIR
        / (
            f"vanilla_stgcn_test_"
            f"subject_{test_subject}.pt"
        )
    )

    cpu_state_dict = {
        name: value.detach().cpu()
        for name, value
        in final_model.state_dict().items()
    }

    torch.save(
        {
            "model_state_dict": (
                cpu_state_dict
            ),
            "test_subject": test_subject,
            "final_epochs": final_epochs,
            "input_channels": 3,
            "num_joints": 33,
            "num_frames": 32,
            "dropout": DROPOUT,
            "learning_rate": (
                LEARNING_RATE
            ),
            "weight_decay": (
                WEIGHT_DECAY
            ),
            "adjacency": ADJACENCY,
        },
        model_path,
    )

    # Lưu checkpoint kết quả sau mỗi outer fold
    pd.DataFrame(
        inner_result_records
    ).to_csv(
        OUTPUT_DIR
        / "inner_results.csv",
        index=False,
    )

    pd.DataFrame(
        outer_result_records
    ).to_csv(
        OUTPUT_DIR
        / "outer_results.csv",
        index=False,
    )

    pd.DataFrame(
        prediction_records
    ).to_csv(
        OUTPUT_DIR
        / "predictions.csv",
        index=False,
    )

    print(
        f"Test S{test_subject}"
        f" | accuracy "
        f"{test_metrics['accuracy']:.4f}"
        f" | balanced accuracy "
        f"{test_metrics['balanced_accuracy']:.4f}"
        f" | sensitivity "
        f"{test_metrics['sensitivity']:.4f}"
        f" | specificity "
        f"{test_metrics['specificity']:.4f}"
    )

    del final_model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("\nĐã hoàn thành 4 outer folds.")

In [ ]:
inner_results = pd.DataFrame(
    inner_result_records
)

outer_results = pd.DataFrame(
    outer_result_records
)

predictions = (
    pd.DataFrame(prediction_records)
    .sort_values("sample_index")
    .reset_index(drop=True)
)


assert len(predictions) == 160
assert predictions["sample_index"].nunique() == 160


metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "sensitivity",
    "specificity",
    "f1",
    "false_alarm_rate",
]


metric_summary = (
    outer_results[metric_columns]
    .agg(["mean", "std"])
    .T
    .reset_index()
    .rename(columns={
        "index": "metric",
    })
)


overall_metrics = calculate_metrics(
    predictions["true_label"],
    predictions["predicted_label"],
)


print("KẾT QUẢ THEO 4 OUTER FOLDS")

display(
    outer_results[
        [
            "test_subject",
            "inner_best_epochs",
            "final_epochs",
            "test_samples",
            *metric_columns,
            "fp",
            "fn",
        ]
    ].round(4)
)


print("\nTRUNG BÌNH 4 OUTER FOLDS")

display(
    metric_summary.round(4)
)


print("\nKẾT QUẢ GỘP 160 VIDEO")

for metric_name in metric_columns:
    print(
        f"{metric_name}:",
        round(
            overall_metrics[
                metric_name
            ],
            4,
        ),
    )


matrix = confusion_matrix(
    predictions["true_label"],
    predictions["predicted_label"],
    labels=[0, 1],
)


display_object = ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=["ADL", "FALL"],
)

display_object.plot(
    cmap="Greens",
    values_format="d",
)

plt.title(
    "Vanilla ST-GCN — Nested LOSO"
)

plt.tight_layout()


CONFUSION_PATH = (
    OUTPUT_DIR
    / "vanilla_stgcn_confusion_matrix.png"
)

plt.savefig(
    CONFUSION_PATH,
    dpi=200,
    bbox_inches="tight",
)

plt.show()


# Danh sách video dự đoán sai
error_videos = predictions.loc[
    ~predictions["correct"]
].copy()


display(error_videos)

print(
    "Tổng video dự đoán sai:",
    len(error_videos),
)


# Lưu bản kết quả hoàn chỉnh
inner_results.to_csv(
    OUTPUT_DIR
    / "inner_results.csv",
    index=False,
)

outer_results.to_csv(
    OUTPUT_DIR
    / "outer_results.csv",
    index=False,
)

predictions.to_csv(
    OUTPUT_DIR
    / "predictions.csv",
    index=False,
)

metric_summary.to_csv(
    OUTPUT_DIR
    / "metric_summary.csv",
    index=False,
)

error_videos.to_csv(
    OUTPUT_DIR
    / "error_videos.csv",
    index=False,
)


print("\nĐã lưu tại:", OUTPUT_DIR)
print("Models:", MODEL_DIR)

In [ ]:
# Kiểm tra đủ số fold
print("Số inner folds:", len(inner_results))
print("Số outer folds:", len(outer_results))
print("Số dự đoán:", len(predictions))
print(
    "Số video duy nhất:",
    predictions["sample_index"].nunique(),
)


# Kiểm tra epoch được chọn bởi validation
epoch_check = inner_results[
    [
        "test_subject",
        "val_subject",
        "best_epoch",
        "balanced_accuracy",
        "f1",
    ]
].copy()

display(epoch_check.round(4))


print(
    "Số inner folds chạm MAX_EPOCHS:",
    (
        epoch_check["best_epoch"]
        >= MAX_EPOCHS
    ).sum(),
)


# Kiểm tra giá trị lỗi
print(
    "Có NaN trong inner results:",
    inner_results.isna().any().any(),
)

print(
    "Có NaN trong outer results:",
    outer_results.isna().any().any(),
)


assert len(inner_results) == 12
assert len(outer_results) == 4
assert len(predictions) == 160
assert predictions["sample_index"].nunique() == 160